# 06 – Fine-tune XLS-R theo LP-FT, chỉ giữ 12 tầng đầu

**Vì sao notebook 05 thất bại** (test 40,7% < đặc trưng đóng băng 49,2%):
1. Head khởi tạo ngẫu nhiên nên gradient những bước đầu phá biểu diễn sẵn có của encoder. Accuracy valid ở epoch 1 chỉ 15%. Đây là hiện tượng *feature distortion* (Kumar et al., ICLR 2022).
2. Chưa hội tụ: accuracy valid vẫn tăng ở epoch 6, loss train 2,39 trong khi mức sàn của LS-0.3 là khoảng 1,83.

**Cách sửa:**
- **LP-FT**: train head trên đặc trưng đóng băng trước (vài giây), sau đó mới fine-tune toàn bộ với learning rate nhỏ.
- **Chỉ giữ 12 tầng transformer đầu.** 73% trọng số weighted-sum nằm ở các tầng 0–12, và tầng tốt nhất là tầng 4–5. Cắt bớt giúp chi phí giảm khoảng một nửa và ít nguy cơ quên kiến thức pre-train hơn.
- Train lâu hơn (12 epoch) với lịch learning rate dạng cosine.

**Kiểm tra tính đúng (sanity check):** ở epoch 0 (chưa fine-tune), accuracy valid phải xấp xỉ accuracy của head LP. Nếu lệch nhiều thì cách tính đặc trưng giữa hai bước không khớp.

- **Runtime:** GPU lớn (A100, RTX PRO 6000) khoảng 1–1,5 giờ cho 2 hàm mục tiêu; L4 khoảng 3 giờ. Có thể chạy tiếp nếu bị ngắt.
- **Kết quả:** `results/lpft_report.json`. Gửi file này cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL = "province_code"
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"
KEEP_LAYERS = 12
LOSSES = None              # None = lấy LS & Geo-LS đã chọn ở notebook 04
EPOCHS = 12
TRAIN_SEC, EVAL_SEC = 6.0, 12.0
BATCH = 16
LR_ENCODER, LR_HEAD, WARMUP = 2e-5, 2e-4, 0.1
MASK_TIME_PROB = 0.05
WORKERS = 4
SEED = 0
SAVE_RESUME = True

In [ ]:
import os, io, glob, json, time, shutil, warnings, multiprocessing as mp
import numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, top_k_accuracy_score
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
LOCAL = "/content" if os.path.isdir("/content") else os.path.abspath("_work")
LOCAL_DATA, FEAT_DIR = f"{LOCAL}/vimd16k", f"{DRIVE_ROOT}/features"
RES_DIR, CKPT_DIR = f"{DRIVE_ROOT}/results", f"{DRIVE_ROOT}/ckpt"
for d in (LOCAL_DATA, RES_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else None
autocast = lambda: torch.autocast(device_type=device.type, dtype=AMP or torch.bfloat16, enabled=AMP is not None)
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "| AMP:", AMP)

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt = y[te]

COORD = {
 "CaoBang": (22.666, 106.258), "LangSon": (21.853, 106.761), "QuangNinh": (20.951, 107.080), "HaiPhong": (20.845, 106.688),
 "ThaiBinh": (20.446, 106.336), "NamDinh": (20.420, 106.168), "PhuTho": (21.322, 105.402), "ThaiNguyen": (21.594, 105.848),
 "YenBai": (21.705, 104.875), "TuyenQuang": (21.823, 105.214), "HaGiang": (22.823, 104.984), "LaoCai": (22.486, 103.971),
 "LaiChau": (22.396, 103.458), "SonLa": (21.326, 103.917), "DienBien": (21.386, 103.023), "HoaBinh": (20.817, 105.338),
 "HaNoi": (21.028, 105.854), "HaiDuong": (20.940, 106.333), "NinhBinh": (20.251, 105.975), "VinhPhuc": (21.310, 105.597),
 "HungYen": (20.646, 106.051), "HaNam": (20.541, 105.914), "BacKan": (22.147, 105.835), "BacGiang": (21.273, 106.195),
 "BacNinh": (21.186, 106.076), "ThanhHoa": (19.807, 105.776), "NgheAn": (18.679, 105.682), "HaTinh": (18.343, 105.906),
 "QuangBinh": (17.468, 106.622), "QuangTri": (16.816, 107.100), "ThuaThienHue": (16.464, 107.586), "DaNang": (16.054, 108.202),
 "QuangNam": (15.573, 108.474), "QuangNgai": (15.120, 108.792), "BinhDinh": (13.776, 109.224), "PhuYen": (13.096, 109.309),
 "KhanhHoa": (12.239, 109.197), "NinhThuan": (11.565, 108.988), "BinhThuan": (10.928, 108.102), "KonTum": (14.350, 108.000),
 "GiaLai": (13.983, 108.000), "DakLak": (12.667, 108.038), "DakNong": (12.004, 107.690), "LamDong": (11.940, 108.458),
 "HoChiMinh": (10.776, 106.701), "DongNai": (10.957, 106.843), "BinhDuong": (10.980, 106.651), "BaRiaVungTau": (10.496, 107.169),
 "TayNinh": (11.310, 106.098), "BinhPhuoc": (11.535, 106.883), "LongAn": (10.535, 106.413), "TienGiang": (10.360, 106.360),
 "BenTre": (10.241, 106.376), "VinhLong": (10.254, 105.972), "TraVinh": (9.935, 106.345), "CanTho": (10.045, 105.747),
 "HauGiang": (9.784, 105.470), "SocTrang": (9.603, 105.980), "BacLieu": (9.294, 105.727), "CaMau": (9.176, 105.150),
 "DongThap": (10.460, 105.633), "AnGiang": (10.386, 105.435), "KienGiang": (10.012, 105.081),
}
LAT = np.array([COORD[n][0] for n in NAMES]); LON = np.array([COORD[n][1] for n in NAMES])
p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))
EYE = torch.eye(C, device=device); DIST_t = torch.tensor(DIST, dtype=torch.float32, device=device)

def target_matrix(kind, p):
    if kind == "ce":
        return EYE
    if kind == "ls":
        return (1 - p) * EYE + p / C
    if kind == "geols":
        eps, tau = p
        nb = torch.exp(-DIST_t / tau) * (1 - EYE)
        return (1 - eps) * EYE + eps * nb / nb.sum(1, keepdim=True)
    raise ValueError(kind)

def metrics(P, yy):
    pred = P.argmax(1); d = DIST[yy, pred]
    return dict(acc=(pred == yy).mean(), macro_f1=f1_score(yy, pred, average="macro"),
                top5=top_k_accuracy_score(yy, P, k=min(5, C - 1), labels=np.arange(C)),
                region_acc=(region_of[pred] == region_of[yy]).mean(), mean_err_km=d.mean(),
                within_100km=(d <= 100).mean(), expected_km=(P * DIST[yy]).sum(1).mean())

if LOSSES is None:
    ch = json.load(open(f"{RES_DIR}/geo_ls_report.json"))["chosen"]
    LOSSES = [("ls", float(ch["ls"])), ("geols", tuple(float(v) for v in ch["geols"].strip("()").split(",")))]
tag = lambda k, p: f"lpft{KEEP_LAYERS}_{k}_" + str(p).replace(" ", "").replace("(", "").replace(")", "").replace(",", "_")
print("Hàm mục tiêu:", LOSSES)

## 1. Linear probing (LP): train head trên đặc trưng đóng băng đã cache
So sánh thêm head dùng 13 tầng (0–12) với head dùng đủ 25 tầng, để kiểm chứng việc cắt bớt mô hình là hợp lý.

In [ ]:
fdir = sorted(glob.glob(f"{FEAT_DIR}/xlsr_*"))[0]
GROUPS = {s: (g.index.values, g["row"].values) for s, g in meta.groupby("shard")}
first = np.load(f"{fdir}/{next(iter(GROUPS))}.npy")
XL = np.zeros((len(meta),) + first.shape[1:], np.float16)
for s, (idx, _) in GROUPS.items():
    XL[idx] = np.load(f"{fdir}/{s}.npy")
D = XL.shape[2]

class Head(nn.Module):
    """chuẩn hoá từng tầng (buffer cố định) → weighted-sum → MLP."""
    def __init__(s, mu, sd):
        super().__init__()
        s.register_buffer("mu", mu.clone()); s.register_buffer("sd", sd.clone())
        s.w = nn.Parameter(torch.zeros(len(mu)))
        s.mlp = nn.Sequential(nn.Linear(D, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
    def forward(s, pooled):                       # pooled: B, L, D (chưa chuẩn hoá)
        z = (pooled.float() - s.mu) / s.sd
        return s.mlp((z * s.w.softmax(0)[None, :, None]).sum(1))

def lp_train(nlayers, Q, seed=SEED, epochs=80, lr=1e-3, wd=1e-3, bs=256, patience=10):
    X = torch.tensor(XL[:, :nlayers], device=device)
    Xtr_, Xva_, Xte_ = [X[torch.tensor(m, device=device)] for m in (tr, va, te)]
    mu, sd = Xtr_.float().mean(0), Xtr_.float().std(0) + 1e-5
    torch.manual_seed(seed)
    h = Head(mu, sd).to(device); opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=wd)
    ytr_ = torch.tensor(y[tr], device=device)
    pred = lambda Z: torch.cat([h(Z[i:i + 2048]).softmax(-1) for i in range(0, len(Z), 2048)]).detach().cpu().numpy()
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        h.train(); perm = torch.randperm(len(Xtr_), device=device)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = -(Q[ytr_[b]] * F.log_softmax(h(Xtr_[b]), -1)).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        h.eval()
        with torch.no_grad():
            acc = (pred(Xva_).argmax(1) == y[va]).mean()
        if acc > best:
            best, state, bad = acc, {k: v.clone() for k, v in h.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    h.load_state_dict(state); h.eval()
    with torch.no_grad():
        mv, mt = metrics(pred(Xva_), y[va]), metrics(pred(Xte_), yt)
    del X, Xtr_, Xva_, Xte_
    return {k: v.cpu() for k, v in h.state_dict().items()}, mv, mt

LP = {}
for kind, p in LOSSES:
    Q = target_matrix(kind, p)
    for nl in (KEEP_LAYERS + 1, XL.shape[1]):
        st, mv, mt = lp_train(nl, Q)
        LP[f"{kind}_{nl}layers"] = dict(valid=mv, test=mt)
        print(f"LP {kind} {p} | {nl} tầng: valid acc {mv['acc']:.4f} | test acc {mt['acc']:.4f}, err {mt['mean_err_km']:.0f} km")
        if nl == KEEP_LAYERS + 1:
            torch.save(st, f"{LOCAL}/{tag(kind, p)}_lp_head.pt")
del XL
torch.cuda.empty_cache()

## 2. Waveform memmap (dùng chung cách làm với notebook 05)

In [ ]:
for f in sorted(glob.glob(f"{DRIVE_ROOT}/data16k/*.parquet")):
    dst = f"{LOCAL_DATA}/{os.path.basename(f)}"
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
        shutil.copy(f, dst)
MAXS = int(EVAL_SEC * 16000)
WAV_PATH, LEN_PATH = f"{LOCAL}/vimd_wave_{int(EVAL_SEC)}s.int16", f"{LOCAL}/vimd_wave_{int(EVAL_SEC)}s.len.npy"

def fill_shard(shard):
    idx, rows = GROUPS[shard]
    mm = np.memmap(WAV_PATH, dtype=np.int16, mode="r+", shape=(len(meta), MAXS))
    col = pq.read_table(f"{LOCAL_DATA}/{shard}", columns=["audio"]).column(0)
    out = []
    for mi, r in zip(idx, rows):
        w = sf.read(io.BytesIO(col[int(r)].as_py()), dtype="int16")[0][:MAXS]
        mm[mi, :len(w)] = w; out.append((mi, len(w)))
    mm.flush()
    return out

if not os.path.exists(LEN_PATH):
    t = time.time()
    np.memmap(WAV_PATH, dtype=np.int16, mode="w+", shape=(len(meta), MAXS)).flush()
    LENS = np.zeros(len(meta), np.int64)
    with mp.get_context("fork").Pool(os.cpu_count()) as pool:
        for res in pool.imap_unordered(fill_shard, list(GROUPS)):
            for mi, n in res:
                LENS[mi] = n
    np.save(LEN_PATH, LENS)
    print(f"Tạo memmap xong sau {time.time() - t:.0f}s")
LENS = np.load(LEN_PATH)

_MM = None
def wave(i):
    global _MM
    if _MM is None:
        _MM = np.memmap(WAV_PATH, dtype=np.int16, mode="r", shape=(len(meta), MAXS))
    return _MM[i, :LENS[i]]

class DS(torch.utils.data.Dataset):
    def __init__(s, idx, train):
        s.idx, s.train = np.asarray(idx), train
    def __len__(s):
        return len(s.idx)
    def __getitem__(s, k):
        i = s.idx[k]; w = wave(i)
        if s.train:
            n = int(TRAIN_SEC * 16000)
            if len(w) > n:
                st = np.random.randint(0, len(w) - n + 1); w = w[st:st + n]
        return w.astype(np.float32) / 32768.0, y[i], k

def collate(batch):
    L = max(len(b[0]) for b in batch)
    x = np.zeros((len(batch), L), np.float32); m = np.zeros((len(batch), L), np.int64)
    for j, (w, _, _) in enumerate(batch):
        w = (w - w.mean()) / (w.std() + 1e-7)
        x[j, :len(w)] = w; m[j, :len(w)] = 1
    return (torch.from_numpy(x), torch.from_numpy(m), torch.tensor([b[1] for b in batch]),
            torch.tensor([b[2] for b in batch]))

def loader(mask, train):
    idx = np.where(mask)[0]
    if not train:
        idx = idx[np.argsort(-LENS[idx])]
    return torch.utils.data.DataLoader(DS(idx, train), batch_size=BATCH, shuffle=train, num_workers=WORKERS,
                                       collate_fn=collate, drop_last=train, pin_memory=device.type == "cuda"), idx

## 3. Fine-tune (FT) từ head LP

In [ ]:
from transformers import AutoModel, get_cosine_schedule_with_warmup

class DialectNet(nn.Module):
    def __init__(s, head_state):
        super().__init__()
        s.enc = AutoModel.from_pretrained(XLSR_MODEL, num_hidden_layers=KEEP_LAYERS, layerdrop=0.0,
                                          mask_time_prob=MASK_TIME_PROB, mask_time_length=10, mask_feature_prob=0.0)
        if hasattr(s.enc.encoder, "layer_norm") and getattr(s.enc.config, "do_stable_layer_norm", False):
            s.enc.encoder.layer_norm = nn.Identity()   # để tầng cuối khớp với đặc trưng cache (chưa qua LayerNorm cuối)
        s.enc.freeze_feature_encoder()
        if device.type == "cuda":
            s.enc.gradient_checkpointing_enable()
        s.head = Head(head_state["mu"], head_state["sd"]); s.head.load_state_dict(head_state)
    def forward(s, x, m):
        out = s.enc(x, attention_mask=m, output_hidden_states=True)
        hs = out.hidden_states[:KEEP_LAYERS + 1]
        fm = s.enc._get_feature_vector_attention_mask(hs[0].shape[1], m).float()
        pooled = torch.stack([(h.float() * fm[..., None]).sum(1) / fm.sum(1, keepdim=True) for h in hs], 1)
        return s.head(pooled)

@torch.no_grad()
def predict(model, mask):
    dl, idx = loader(mask, False); model.eval()
    P = np.zeros((len(idx), C), np.float32)
    for x, m, _, k in dl:
        with autocast():
            P[k.numpy()] = model(x.to(device, non_blocking=True), m.to(device)).float().softmax(-1).cpu().numpy()
    out = np.zeros((mask.sum(), C), np.float32)
    out[np.searchsorted(np.where(mask)[0], idx)] = P
    return out

def run(kind, p):
    name = tag(kind, p); ck = f"{CKPT_DIR}/{name}_last.pt"
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = DialectNet(torch.load(f"{LOCAL}/{name}_lp_head.pt")).to(device)
    enc_params = [q for q in model.enc.parameters() if q.requires_grad]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER},
                             {"params": list(model.head.parameters()), "lr": LR_HEAD}], weight_decay=0.01)
    dl_tr, _ = loader(tr, True)
    steps = EPOCHS * len(dl_tr)
    sched = get_cosine_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler(device.type, enabled=AMP == torch.float16)
    Q = target_matrix(kind, p)
    st = dict(epoch=0, best_valid=-1.0, history=[])
    if SAVE_RESUME and os.path.exists(ck):
        s = torch.load(ck, map_location="cpu", weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        st = s["state"]; print(f"[{name}] chạy tiếp từ epoch {st['epoch']}")

    def evaluate(ep, t, tot):
        Pv = predict(model, va); mv = metrics(Pv, y[va])
        rec = dict(epoch=ep, train_loss=tot, **{f"valid_{k}": float(v) for k, v in mv.items()}, minutes=(time.time() - t) / 60)
        if mv["acc"] > st["best_valid"]:
            st["best_valid"] = float(mv["acc"])
            Pt = predict(model, te); np.save(f"{RES_DIR}/{name}_test_probs.npy", Pt)
            st["best"] = dict(epoch=ep, **{f"test_{k}": float(v) for k, v in metrics(Pt, yt).items()})
            rec["new_best"] = True
        st["history"].append(rec)
        print(f"[{name}] epoch {ep}: valid acc {mv['acc']:.4f}, err {mv['mean_err_km']:.0f} km | best test acc "
              f"{st['best']['test_acc']:.4f} (epoch {st['best']['epoch']}) | {rec['minutes']:.1f} phút", flush=True)

    if st["epoch"] == 0 and not st["history"]:
        evaluate(0, time.time(), None)          # sanity check: phải ≈ valid acc của LP 13 tầng
    for ep in range(st["epoch"], EPOCHS):
        model.train(); t = time.time(); tot = 0.0
        for i, (x, m, yb, _) in enumerate(dl_tr):
            x, m, yb = x.to(device, non_blocking=True), m.to(device), yb.to(device)
            with autocast():
                logits = model(x, m)
            loss = -(Q[yb] * F.log_softmax(logits.float(), -1)).sum(1).mean()
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            tot += loss.item()
            if i % 200 == 0:
                el = time.time() - t
                print(f"  [{name}] ep {ep + 1} step {i}/{len(dl_tr)} loss {tot / (i + 1):.3f} "
                      f"| còn ~{el / (i + 1) * (len(dl_tr) - i - 1) / 60:.0f} phút epoch này", flush=True)
        st["epoch"] = ep + 1
        evaluate(ep + 1, t, tot / len(dl_tr))
        if SAVE_RESUME and ep + 1 < EPOCHS:
            tmp = f"{LOCAL}/{name}_last.pt"
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), state=st), tmp)
            shutil.copy(tmp, ck + ".part"); os.replace(ck + ".part", ck); os.remove(tmp)
    json.dump(st, open(f"{RES_DIR}/{name}_state.json", "w"), indent=1, default=float)
    if os.path.exists(ck):
        os.remove(ck)
    del model, opt; torch.cuda.empty_cache()
    return st

RESULTS = {}
for kind, p in LOSSES:
    name = tag(kind, p)
    if os.path.exists(f"{RES_DIR}/{name}_state.json"):
        RESULTS[name] = json.load(open(f"{RES_DIR}/{name}_state.json")); print(f"[{name}] đã xong trước đó")
    else:
        RESULTS[name] = run(kind, p)

## 4. Tổng hợp

In [ ]:
B = np.random.default_rng(0).integers(0, len(yt), (2000, len(yt)))
def paired(a, b):
    dacc = (a == yt).astype(float) - (b == yt); dkm = DIST[yt, a] - DIST[yt, b]
    return dict(acc_diff=round(float(dacc.mean()), 4), acc_ci95=np.percentile(dacc[B].mean(1), [2.5, 97.5]).round(4).tolist(),
                km_diff=round(float(dkm.mean()), 1), km_ci95=np.percentile(dkm[B].mean(1), [2.5, 97.5]).round(1).tolist())
PRED = {n: np.load(f"{RES_DIR}/{n}_test_probs.npy").argmax(1) for n in RESULTS}
if os.path.exists(f"{RES_DIR}/pred_xlsr_wsum_mlp.npy"):
    PRED["frozen_xlsr_wsum_mlp"] = np.load(f"{RES_DIR}/pred_xlsr_wsum_mlp.npy")
names = list(PRED)
COMPARE = {f"{a} - {b}": paired(PRED[a], PRED[b]) for i, a in enumerate(names) for b in names[i + 1:]}
print(pd.DataFrame({n: s["best"] for n, s in RESULTS.items()}).T.round(4).to_string())
print(json.dumps(COMPARE, indent=1))
report = dict(lp=LP, results=RESULTS, compare=COMPARE, losses=[[k, str(p)] for k, p in LOSSES],
              config=dict(model=XLSR_MODEL, keep_layers=KEEP_LAYERS, epochs=EPOCHS, train_sec=TRAIN_SEC, eval_sec=EVAL_SEC,
                          batch=BATCH, lr_encoder=LR_ENCODER, lr_head=LR_HEAD, mask_time_prob=MASK_TIME_PROB),
              gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
              total_minutes=round((time.time() - T0) / 60, 1))
json.dump(report, open(f"{RES_DIR}/lpft_report.json", "w"), ensure_ascii=False, indent=1, default=float)
print("Đã lưu results/lpft_report.json – gửi cho Claude.")